# 02 - Auditoria dos dados brutos
Objetivo: medir quantos registros cada tipo de problema afeta, por arquivo e por coluna. Esses números são o "antes" que comprova a limpeza e formam o dicionário de inconsistências do README.


In [1]:
import pandas as pd

compras = pd.read_csv("../data/raw/compras.csv", dtype=str)
itens = pd.read_csv("../data/raw/itens.csv", dtype=str)
resultados = pd.read_csv("../data/raw/resultados.csv", dtype=str)


---
## 1. Documentos (CNPJ/CPF)


In [2]:
def classificar_documento(doc, tipo):
    if pd.isna(doc):
        return "vazio"
    if tipo == "PE":
        return "estrangeiro"
    tamanho = 14 if tipo == "PJ" else 11
    if doc != doc.strip():
        return "espaços extras"
    if not doc.isdigit():
        return "máscara"
    if len(doc) < tamanho:
        return "zeros perdidos"
    if len(doc) == tamanho:
        return "limpo"
    return "dígitos demais"


In [3]:
compras["cnpj_orgao"].apply(lambda doc: classificar_documento(doc, "PJ")).value_counts()


cnpj_orgao
limpo             1658
máscara            176
espaços extras     131
zeros perdidos      77
Name: count, dtype: int64

In [4]:
situacao_ni = [classificar_documento(doc, tipo)
               for doc, tipo in zip(resultados["ni_fornecedor"], resultados["tipo_pessoa"])]

pd.crosstab(pd.Series(situacao_ni, name="situação"), resultados["tipo_pessoa"], margins=True)


tipo_pessoa,PE,PF,PJ,All
situação,,,,
espaços extras,0,293,695,988
estrangeiro,21,0,0,21
limpo,0,3687,9216,12903
máscara,0,392,1239,1631
zeros perdidos,0,159,194,353
All,21,4531,11344,15896


**Resultado:**

| Coluna | Limpo | Máscara | Espaços extras | Zeros perdidos | Estrangeiro | Total |
|---|---|---|---|---|---|---|
| `cnpj_orgao` | 1.658 | 176 | 131 | 77 | – | 2.042 |
| `ni_fornecedor` (PJ) | 9.216 | 1.239 | 695 | 194 | – | 11.344 |
| `ni_fornecedor` (PF) | 3.687 | 392 | 293 | 159 | – | 4.531 |
| `ni_fornecedor` (PE) | – | – | – | – | 21 | 21 |

19% dos documentos de fornecedor e 19% dos CNPJs de órgão têm algum problema. Sem a limpeza, o mesmo fornecedor apareceria com documentos diferentes: no bruto há 11.418 documentos distintos, e após padronizar sobram 10.299.


---
## 2. Valores monetários e numéricos


In [5]:
def auditar_numero(coluna):
    pontos = coluna.str.count(r"\.")
    virgulas = coluna.str.count(",")
    return pd.Series({
        "vazio": coluna.isna().sum(),
        "já no formato numérico (ex.: 3.6)": coluna.str.fullmatch(r"\d+(\.\d+)?").sum(),
        "com R$": coluna.str.contains(r"R\$", na=False).sum(),
        "espaços extras ou invisíveis": coluna.str.contains(r"^\s|\s$|\xa0", na=False).sum(),
        "negativo entre parênteses": coluna.str.contains(r"\(", na=False).sum(),
        "vírgula como decimal (ex.: 3,54)": ((virgulas == 1) & (pontos == 0)).sum(),
        "separador de milhar": ((pontos > 1) | (virgulas > 1) | ((pontos > 0) & (virgulas > 0))).sum(),
    })


In [6]:
auditoria_numeros = pd.DataFrame({
    "itens.quantidade": auditar_numero(itens["quantidade"]),
    "itens.valor_unitario_estimado": auditar_numero(itens["valor_unitario_estimado"]),
    "itens.valor_total_estimado": auditar_numero(itens["valor_total_estimado"]),
    "resultados.quantidade_homologada": auditar_numero(resultados["quantidade_homologada"]),
    "resultados.valor_unitario_homologado": auditar_numero(resultados["valor_unitario_homologado"]),
    "resultados.valor_total_homologado": auditar_numero(resultados["valor_total_homologado"]),
}).T

auditoria_numeros


,vazio,já no formato numérico (ex.: 3.6),com R$,espaços extras ou invisíveis,negativo entre parênteses,"vírgula como decimal (ex.: 3,54)",separador de milhar
itens.quantidade,0,5596,1148,551,0,1262,219
itens.valor_unitario_estimado,0,5475,1188,561,0,953,649
itens.valor_total_estimado,0,5415,1171,588,0,359,1303
resultados.quantidade_homologada,0,12472,2561,1267,0,2245,1179
resultados.valor_unitario_homologado,0,12282,2650,1352,0,2077,1537
resultados.valor_total_homologado,0,12064,2647,1332,0,690,3142


**Resultado:** cada coluna da tabela conta uma característica de forma independente. Um mesmo valor pode ter várias (ex.: `" R$ 1.234,56"` tem espaço, `R$` e milhar), por isso a soma de uma linha passa do total de registros.

- Cerca de 1 em cada 4 valores **não** está no formato numérico do Python e precisa de conversão.
- `R$` aparece também nas **quantidades** (1.148 em `itens` e 2.561 em `resultados`): formatação de moeda aplicada em campo errado no sistema de origem.
- Há espaços invisíveis (`\xa0`), vírgula decimal e separadores de milhar no padrão brasileiro e no americano em todas as 6 colunas.
- **Não há** valores negativos entre parênteses nesta base, embora o enunciado cite o caso.
- Nenhum valor vazio nas 6 colunas.

Regra de conversão (regra 12): um símbolo sozinho é sempre o decimal (`746.815` = 746,815); símbolo repetido é milhar; com os dois, o último é o decimal.


---
## 3. Datas


In [7]:
formato_publicacao = compras["data_publicacao"].str.replace(r"\d", "9", regex=True).value_counts()
formato_resultado = resultados["data_resultado"].str.replace(r"\d", "9", regex=True).value_counts()

pd.DataFrame({"data_publicacao": formato_publicacao, "data_resultado": formato_resultado}).fillna(0).astype(int)


,data_publicacao,data_resultado
99-99-9999,138,912
99.99.9999,120,937
99/99/9999,117,929
99/99/9999 99:99,126,934
9999-99-99,0,10301
9999-99-99T99:99:99,1319,0
9999/99/99,116,960
99999999,106,923


In [8]:
from datetime import datetime

FORMATOS_DATA = ["%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y", "%d.%m.%Y", "%Y/%m/%d", "%Y%m%d"]

def converter_data(texto):
    if pd.isna(texto):
        return None
    texto = texto.strip()[:10]
    for formato in FORMATOS_DATA:
        try:
            return datetime.strptime(texto, formato).date()
        except ValueError:
            continue
    return None

compras["publicacao"] = compras["data_publicacao"].map(converter_data)
resultados["homologacao"] = resultados["data_resultado"].map(converter_data)

print("Publicações não convertidas:", compras["publicacao"].isna().sum())
print("Homologações não convertidas:", resultados["homologacao"].isna().sum())


Publicações não convertidas: 0
Homologações não convertidas: 0


In [9]:
datas = resultados.merge(
    compras[["numero_controle_pncp", "publicacao"]],
    on="numero_controle_pncp",
    how="left",
)

antes = datas["homologacao"] < datas["publicacao"]
mesmo_dia = datas["homologacao"] == datas["publicacao"]

print("Resultados:", len(datas))
print("Homologação ANTES da publicação (impossível):", antes.sum())
print("Homologação no MESMO dia da publicação (válida, ciclo 0):", mesmo_dia.sum())


Resultados: 15896
Homologação ANTES da publicação (impossível): 234
Homologação no MESMO dia da publicação (válida, ciclo 0): 1239


**Resultado:**

**Formatos:** 7 formatos em cada coluna. O ISO aparece com hora na publicação (`2025-01-06T11:51:06`) e sem hora no resultado (`2025-03-10`); ao descartar a hora (10 primeiros caracteres), os dois se igualam. Com a lista de 6 formatos, 100% das datas foram convertidas (nenhuma falha).

**Cronologia (regra 7):** dos 15.896 resultados,
- 234 têm homologação **anterior** à publicação: data impossível. Na fato, o item continua (conta no volume), mas fica sem data de homologação e sem ciclo;
- 1.239 foram homologados no **mesmo dia** da publicação: válidos, com ciclo de 0 dias.

Esta contagem é por resultado (todos os classificados). Na fato, com um vencedor por item, o número de itens sem ciclo é diferente.
